# PHYS690 - Lecture 13: Hermitian Quantum Eigensystems
## Anharmonic oscillators, general Hermitian matrices, and near-degeneracy

**Primary physics example:** numerical diagonalization of the one-dimensional quartic oscillator,

$$
\hat H=-\frac{1}{2}\frac{d^2}{dq^2}
+\frac{m^2}{2}q^2+\frac{\lambda}{4}q^4,
$$

in dimensionless units with $\hbar=1$ and particle mass $m=1$.

## Learning goals

By the end of class you should be able to:

- connect the quantum eigenvalue boundary-value problem to the forced boundary-value problems from Lectures 11 and 12;
- convert the time-independent Schrodinger equation into a Hermitian matrix eigenproblem;
- impose a finite computational domain and explain the boundary approximation it introduces;
- use `scipy.linalg.eigh` to solve a general dense Hermitian eigenvalue problem;
- normalize discrete eigenvectors as continuum wavefunctions;
- validate eigenpairs using residuals, orthogonality, parity, and the harmonic-oscillator limit;
- separate finite-domain error from finite-grid-spacing error;
- distinguish Hermiticity, sparsity, bandedness, and tridiagonality as separate matrix properties;
- explain why near-degenerate eigenvectors are sensitive to small perturbations even when their eigenspace is stable;
- connect the oscillator and double-well examples to concepts encountered in introductory QFT.

In [ ]:
# Arrays, tables, and plotting support the numerical workflow.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use the structure-aware solver for dense Hermitian matrices.
from scipy.linalg import eigh
# Hermite polynomials provide the exact harmonic-oscillator benchmark.
from scipy.special import eval_hermite, factorial

# Keep figures visually consistent with the rest of the course notebooks.
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True})

## 1. The quantum eigenvalue problem

For a particle moving in one dimension,

$$
\hat H\psi_n(q)=E_n\psi_n(q),
\qquad
\hat H=-\frac{1}{2}\frac{d^2}{dq^2}+V(q).
$$

The Hamiltonian is self-adjoint on its physical domain. Consequently:

- its eigenvalues are real;
- eigenstates with distinct energies are orthogonal;
- its eigenfunctions form a basis under appropriate conditions.

For bound states on the infinite line, $\psi(q)\rightarrow0$ as $|q|\rightarrow\infty$. A computer cannot represent the infinite line, so we choose a finite box $[-q_{\max},q_{\max}]$ and impose

$$
\psi(-q_{\max})=\psi(q_{\max})=0.
$$

This is a controlled approximation only if the wavefunctions of interest are already negligible near both boundaries.

### From the differential equation to a finite-difference Hamiltonian

Begin with the time-independent Schrodinger equation written without operator shorthand:

$$
-\frac{1}{2}\frac{d^2\psi(q)}{dq^2}+V(q)\psi(q)=E\psi(q).
$$

Divide $[-q_{\max},q_{\max}]$ into $N$ equal intervals. The grid points and spacing are

$$
q_i=-q_{\max}+ih,\qquad h=\frac{2q_{\max}}{N},\qquad i=0,1,\ldots,N.
$$

Define $\psi_i=\psi(q_i)$ and $V_i=V(q_i)$. At each interior point $i=1,\ldots,N-1$, the differential equation is

$$
-\frac{1}{2}\psi''(q_i)+V_i\psi_i=E\psi_i.
$$

Taylor-expand the wavefunction one grid point to either side:

$$
\psi(q_i\pm h)=\psi_i\pm h\psi'_i+\frac{h^2}{2}\psi''_i
\pm\frac{h^3}{6}\psi'''_i+\mathcal{O}(h^4).
$$

Adding the $+$ and $-$ expansions cancels the odd derivatives. Solving for the second derivative gives the centered approximation

$$
\psi''(q_i)=\frac{\psi_{i-1}-2\psi_i+\psi_{i+1}}{h^2}
+\mathcal{O}(h^2).
$$

Substitute this expression into the Schrodinger equation:

$$
-\frac{1}{2}\left(\frac{\psi_{i-1}-2\psi_i+\psi_{i+1}}{h^2}\right)
+V_i\psi_i=E\psi_i.
$$

Distributing the kinetic-energy factor and grouping coefficients of neighboring wavefunction values produces one matrix row:

$$
-\frac{1}{2h^2}\psi_{i-1}
+\left(\frac{1}{h^2}+V_i\right)\psi_i
-\frac{1}{2h^2}\psi_{i+1}
=E\psi_i.
$$

The Dirichlet conditions set $\psi_0=\psi_N=0$, so only the $N-1$ interior values are unknown. Stacking their equations gives

$$
\underbrace{\begin{pmatrix}
\frac{1}{h^2}+V_1 & -\frac{1}{2h^2} & 0 & \cdots & 0 \\
-\frac{1}{2h^2} & \frac{1}{h^2}+V_2 & -\frac{1}{2h^2} & \ddots & \vdots \\
0 & \ddots & \ddots & \ddots & 0 \\
\vdots & \ddots & -\frac{1}{2h^2} & \frac{1}{h^2}+V_{N-2} & -\frac{1}{2h^2} \\
0 & \cdots & 0 & -\frac{1}{2h^2} & \frac{1}{h^2}+V_{N-1}
\end{pmatrix}}_{H}
\begin{pmatrix}\psi_1\\\psi_2\\\vdots\\\psi_{N-2}\\\psi_{N-1}\end{pmatrix}
=E
\begin{pmatrix}\psi_1\\\psi_2\\\vdots\\\psi_{N-2}\\\psi_{N-1}\end{pmatrix}.
$$

Therefore the differential equation has become the algebraic eigenvalue problem

$$
\boxed{H\boldsymbol\psi=E\boldsymbol\psi}.
$$

The diagonal entries contain the kinetic contribution $1/h^2$ plus the local potential $V_i$; the two off-diagonals contain the nearest-neighbor kinetic coupling $-1/(2h^2)$. This particular representation is real, symmetric, and tridiagonal (because a three-point stencil couples only nearest neighbors). It is the quantum analogue of the finite-difference operator assembly used for the Poisson equation in Lecture 11.

### Using eigenvalue solver `eigh`: BVP workflow with a different algebraic question

This is not a new discretization technology. Lectures 11 and 12 began with a differential operator, imposed boundary conditions, evaluated the operator on a grid, and assembled a matrix. We are doing the same thing here.

The difference is what is known and what must be solved for:

| Problem | Continuous equation | Discrete equation | Unknown | SciPy tool |
|---|---|---|---|---|
| Poisson BVP | $\mathcal{L}u=s$ | $A\mathbf{u}=\mathbf{b}$ | field $\mathbf{u}~(i.e.~V, \vec{E})$ for a known source $\mathbf{b}~(\rho)$  | `spsolve` |
| Quantum eigenvalue BVP | $\hat H\psi=E\psi$ | $H\boldsymbol\psi=E\boldsymbol\psi$ | allowed energies $E_n$ **and** states $\boldsymbol\psi_n$ | `eigh` |

For the poisson BVP, the source vector selects one solution after the boundary conditions are imposed. For the quantum problem there is no external source vector. We seek nonzero solutions of

$$
(H-EI)\boldsymbol\psi=0.
$$

A nonzero solution exists only when

$$
\det(H-EI)=0.
$$

The boundary conditions therefore select discrete allowed energies. We use `scipy.linalg.eigh` because the discretized Hamiltonian is Hermitian:

$$
H=H^\dagger.
$$

For a Hermitian matrix, the eigenvalues are real and the eigenvectors can be chosen orthonormal. `eigh` preserves and exploits that structure; the more general `eig` does not assume it. The option `subset_by_index` also lets us request only the lowest-energy states.

For a dense matrix of dimension $D$, storage grows as $D^2$ and full diagonalization generally grows as $D^3$. For much larger sparse Hamiltonians, `scipy.sparse.linalg.eigsh` targets a few eigenpairs without dense diagonalization. That is a performance refinement of the same eigenvalue BVP, not different physics.

Note: the characteristic equation $\det(H-EI)=0$ explains mathematically why only particular energies are allowed, but `eigh` does **not** evaluate this determinant or search for its zeros. Numerical eigensolvers instead use stable matrix-reduction and iterative algorithms to obtain the eigenvalues and eigenvectors directly; explicitly forming a determinant would generally be less accurate and less efficient. The open [LAPACK Users' Guide section on symmetric eigenproblems](https://www.netlib.org/lapack/lug/node48.html) describes the standard stages: reduce a real-symmetric or complex-Hermitian matrix to real tridiagonal form, solve the tridiagonal eigenproblem, and transform the eigenvectors back to the original basis.


Use the finite-difference row above to predict the matrix structure before reading `build_hamiltonian`.

In [ ]:
def quartic_potential(q, mass_squared=1.0, coupling=0.0):
    """Return V(q) = m^2 q^2 / 2 + lambda q^4 / 4."""
    # The quadratic term gives the harmonic limit; the quartic term adds the interaction.
    return 0.5 * mass_squared * q**2 + 0.25 * coupling * q**4


def build_hamiltonian(q_max, interval_count, potential_function):
    """Construct the full coordinate-basis Hamiltonian matrix."""
    # N intervals require N+1 coordinates when both endpoints are included.
    q_full = np.linspace(-q_max, q_max, interval_count + 1)

    # Read the uniform spacing from the grid rather than recomputing it separately.
    h = q_full[1] - q_full[0]

    # Remove the two Dirichlet boundary values; only interior psi_i are unknown.
    q = q_full[1:-1]

    # Evaluate the selected physical potential at every interior grid point.
    potential = potential_function(q)

    # The centered second derivative contributes 1/h^2 on the diagonal.
    diagonal = np.full(q.size, 1.0 / h**2)

    # Each point couples to its two nearest neighbors with -1/(2h^2).
    nearest_neighbor = np.full(q.size - 1, -0.5 / h**2)
    # Place those coefficients on the three appropriate matrix diagonals.
    kinetic_matrix = (
        np.diag(diagonal)
        + np.diag(nearest_neighbor, k=1)
        + np.diag(nearest_neighbor, k=-1)
    )

    # A local coordinate-space potential is diagonal in the position basis.
    potential_matrix = np.diag(potential)

    # The complete Hamiltonian is the sum of kinetic and potential operators.
    hamiltonian = kinetic_matrix + potential_matrix

    # Check the physical matrix property rather than assuming it from assembly.
    hermiticity_error = np.linalg.norm(
        hamiltonian - hamiltonian.conj().T, ord="fro"
    )
    if hermiticity_error > 1e-12:
        raise ValueError("constructed Hamiltonian is not Hermitian")

    # Keep coordinate-space quantities together for later tables and plots.
    grid_df = pd.DataFrame({
        "q": q,
        "potential": potential,
        "hamiltonian_diagonal": np.diag(hamiltonian),
    })
    # Return the grid metadata beside the matrix it describes.
    return grid_df, hamiltonian, h


def solve_low_energy_states(q_max, interval_count, potential_function, state_count=6):
    """Solve a general dense Hermitian eigenproblem and normalize its states."""
    # Reuse one assembly function so every calculation uses identical conventions.
    grid_df, hamiltonian, h = build_hamiltonian(
        q_max, interval_count, potential_function
    )

    # eigh accepts real symmetric or complex Hermitian matrices. Request only
    # the lowest part of the spectrum needed for this physics problem.
    # Eigenvectors are returned as columns ordered by increasing energy.
    energies, euclidean_vectors = eigh(
        hamiltonian,
        subset_by_index=(0, state_count - 1),
        check_finite=True,
    )

    # eigh gives sum_i |v_i|^2 = 1. Divide by sqrt(h) so that
    # h * sum_i |psi_i|^2 approximates the continuum integral of |psi|^2.
    wavefunctions = euclidean_vectors / np.sqrt(h)

    # Return both normalizations: matrix residuals use v, while integrals use psi.
    return grid_df, hamiltonian, h, energies, euclidean_vectors, wavefunctions

### Discrete and continuum normalization

The eigensolver returns vectors normalized with the Euclidean inner product,

$$
\sum_i |v_i|^2=1.
$$

A sampled wavefunction should instead approximate a sum of discrete steps with size $h$

$$
\int |\psi(q)|^2dq=1
\quad\Longrightarrow\quad
h\sum_i|\psi_i|^2\approx1.
$$

Thus $\psi_i=v_i/\sqrt{h}$. This factor does not change the eigenvalue equation, but it matters for probability densities, expectation values, and overlaps.

## 2. Exact benchmark: the harmonic oscillator

Set $m^2=1$ and $\lambda=0$. The exact dimensionless spectrum and normalized wavefunctions are

$$
E_n=n+\frac{1}{2},
$$

$$
\psi_n(q)=\frac{1}{\sqrt{2^n n!\sqrt{\pi}}}
H_n(q)e^{-q^2/2},
\qquad n=0,1,2,\ldots
$$

$H_n(q)$ is the $n^{th}$ **Hermite polynomial** (not the Hamiltonian). For example, $H_0(q)=1$, $H_1(q)=2q$, and $H_2(q)=4q^2-2$. It supplies the nodes and parity of the $n$th state, while the Gaussian factor $e^{-q^2/2}$ makes the wavefunction decay at large $|q|$.

The harmonic limit tests the complete numerical pipeline: operator signs, kinetic-energy coefficient, boundary conditions, grid spacing, eigenvalue ordering, and normalization.

The physical sign or complex phase of an eigenvector is arbitrary. Compare wavefunctions through an absolute overlap or align their signs before plotting.

### Prediction before coding

Before solving the harmonic oscillator numerically, predict:

> What should the first three energies, parities, and numbers of nodes be? Which predictions test the physical discretization rather than only the eigensolver?

Keep these predictions visible when interpreting the numerical table and wavefunction plots.

In [ ]:
def harmonic_wavefunction(n, q):
    """Evaluate the normalized dimensionless harmonic-oscillator state n."""
    # Evaluate the normalization separately to keep the analytic formula readable.
    normalization = np.sqrt((2.0**n) * factorial(n) * np.sqrt(np.pi))
    return eval_hermite(n, q) * np.exp(-0.5 * q**2) / normalization


# The box is wide enough that the low-energy tails are very small at its edges.
q_max_demo = 7.0
intervals_demo = 500

# Assemble and diagonalize the harmonic Hamiltonian for its six lowest states.
(
    harmonic_grid_df,
    harmonic_hamiltonian,
    h_demo,
    harmonic_energies,
    harmonic_vectors,
    harmonic_wavefunctions,
) = solve_low_energy_states(
    q_max=q_max_demo,
    interval_count=intervals_demo,
    potential_function=lambda q: quartic_potential(q, 1.0, 0.0),
    state_count=6,
)

# Hermiticity is the general structural check; tridiagonality is incidental here.
hermiticity_error = np.linalg.norm(
    harmonic_hamiltonian - harmonic_hamiltonian.conj().T, ord="fro"
)

# Put numerical and analytic energies in the same table for direct comparison.
energy_summary_df = pd.DataFrame({
    "state_n": np.arange(6),
    "energy_numeric": harmonic_energies,
    "energy_exact": np.arange(6) + 0.5,
})
# A signed error shows whether the discretization shifts each level up or down.
energy_summary_df["energy_error"] = (
    energy_summary_df["energy_numeric"] - energy_summary_df["energy_exact"]
)

# Report the structural check before interpreting the spectrum.
print(f"Hermiticity check ||H-H^dagger||_F = {hermiticity_error:.3e}")
energy_summary_df

### Plotting wavefunction solutions

In [ ]:
# Store wavefunctions and probability densities in a DataFrame for plotting.
harmonic_plot_df = harmonic_grid_df[["q", "potential"]].copy()

# Compare the first four numerical states with their exact wavefunctions.
for n in range(4):
    numerical = harmonic_wavefunctions[:, n].copy()
    analytic = harmonic_wavefunction(n, harmonic_plot_df["q"].to_numpy())

    # Align the arbitrary numerical sign with the analytic state.
    if h_demo * np.sum(numerical * analytic) < 0:
        numerical *= -1.0

    # Add each aligned pair as named columns for transparent plotting.
    harmonic_plot_df[f"psi_{n}_numeric"] = numerical
    harmonic_plot_df[f"psi_{n}_exact"] = analytic

# Use one panel per state so nodes and sign changes remain visible.
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, constrained_layout=True)
for n, ax in enumerate(axes.flat):
    # Draw the numerical and exact curves on identical axes.
    ax.plot(
        harmonic_plot_df["q"],
        harmonic_plot_df[f"psi_{n}_numeric"],
        label="finite difference",
    )
    ax.plot(
        harmonic_plot_df["q"],
        harmonic_plot_df[f"psi_{n}_exact"],
        linestyle="--",
        label="analytic",
    )
    ax.set(title=fr"$n={n}$, $E_n={harmonic_energies[n]:.4f}$", ylabel=r"$\psi_n(q)$")

# Apply shared labels and a single legend after filling all panels.
axes[-1, 0].set_xlabel(r"$q$")
axes[-1, 1].set_xlabel(r"$q$")
axes[0, 0].legend()
plt.show()

## In-class Activity 1 - Validate the harmonic eigenpairs (13 minutes)

For the first five states of the full Hermitian matrix:

1. compare the numerical energy with $E_n=n+1/2$;
2. calculate the eigenpair residual;
3. compute the continuum overlap with the analytic state;
4. compute the parity expectation.

For a numerical eigenpair $(E_n,\mathbf{v}_n)$, define the residual vector and its size by

$$
\mathbf{r}_n=H\mathbf{v}_n-E_n\mathbf{v}_n,
\qquad
r_n=\|\mathbf{r}_n\|_2.
$$

Here $\mathbf{v}_n$ is the Euclidean-normalized vector returned by `eigh`. A small $r_n$ shows that the pair satisfies the **discrete matrix equation** accurately. It does not by itself show that the finite box and grid accurately approximate the continuum Hamiltonian; those errors require the convergence tests later in the lecture.

The parity expectation is

$$
\langle P\rangle_n
=\int\psi_n(q)\psi_n(-q)dq
\approx h\sum_i\psi_{n,i}\psi_{n,N-1-i}.
$$

For a symmetric potential, the expected parities are $+1,-1,+1,-1,\ldots$.

**Discuss before coding:** Suppose the finite box is too small, but `eigh` solves that discrete Hamiltonian accurately. Which diagnostic could still be very small: the residual, energy error, analytic overlap, or boundary probability? What does that diagnostic actually validate?


In [ ]:
# Accumulate one set of diagnostics for each low-energy state.
validation_rows = []

for n in range(5):
    # Extract both discrete-vector and continuum-wavefunction normalizations.
    energy = harmonic_energies[n]
    euclidean_vector = harmonic_vectors[:, n]
    wavefunction = harmonic_wavefunctions[:, n]
    analytic = harmonic_wavefunction(n, harmonic_grid_df["q"].to_numpy())

    # TODO: compute the exact harmonic-oscillator energy.
    exact_energy = ...

    # TODO: compute ||H v - E v||_2 using matrix-vector multiplication.
    residual = ...

    # TODO: compute the absolute continuum overlap with the analytic state.
    absolute_overlap = ...

    # TODO: compute the parity expectation by reversing the sampled wavefunction.
    parity_expectation = ...

    # Keep solver, continuum, and symmetry checks in the same output row.
    validation_rows.append({
        "state_n": n,
        "energy_numeric": energy,
        "energy_exact": exact_energy,
        "eigenpair_residual": residual,
        "absolute_overlap": absolute_overlap,
        "parity_expectation": parity_expectation,
    })

# Convert the accumulated dictionaries into a table for comparison across states.
validation_df = pd.DataFrame(validation_rows)
validation_df

## 3. Orthogonality, residuals, and physical validation

For the continuum-normalized numerical states,

$$
h\Psi^\dagger\Psi\approx I.
$$

An eigenpair residual tests whether the solver found an eigenpair of the **discrete matrix**. It does not prove that the matrix accurately represents the continuum Hamiltonian. Agreement with the analytic limit, small boundary probability, and convergence as $h\rightarrow0$ test different parts of the calculation.

This distinction is the same one made earlier in the course between optimizer convergence and model adequacy: a numerical algorithm can solve the problem it was given very accurately even when the formulated problem is inadequate.

In [ ]:
# Verify continuum orthonormality for the first six states.
# The conjugate transpose also works if a future Hamiltonian has complex states.
gram_matrix = h_demo * harmonic_wavefunctions.conj().T @ harmonic_wavefunctions
# Label rows and columns so off-diagonal overlaps are easy to locate.
gram_df = pd.DataFrame(
    gram_matrix,
    index=[f"state {n}" for n in range(6)],
    columns=[f"state {n}" for n in range(6)],
)

# Check how much ground-state probability lies in the outer 10% of the box.
# Select grid points closest to either artificial box boundary.
edge_mask = np.abs(harmonic_grid_df["q"]) > 0.9 * q_max_demo

# Integrate |psi_0|^2 over that edge region using the rectangle rule.
edge_probability = h_demo * np.sum(np.abs(harmonic_wavefunctions[edge_mask, 0]) ** 2)

# Display the boundary diagnostic followed by the overlap matrix.
print(f"Ground-state probability in outer 10% of box: {edge_probability:.3e}")
gram_df.round(10)

## 4. Turn on the quartic interaction

Now use

$$
V(q)=\frac{1}{2}q^2+\frac{\lambda}{4}q^4,
\qquad \lambda=0.4.
$$

The quartic term confines the particle more strongly at large $|q|$ and raises the energy levels, especially the highly excited states. Unlike the harmonic oscillator, this problem has no elementary closed-form spectrum, so convergence and internal diagnostics become the evidence that the numerical answer is trustworthy.

### Prediction before coding

> For positive $\lambda$, should the energies move up or down? Should the interaction-induced shift be larger for low or highly excited states? Does the quartic term preserve parity?

Make a qualitative prediction before computing the interacting spectrum.


In [ ]:
# Turn on a positive quartic coupling while retaining the harmonic term.
quartic_coupling = 0.4
# Package the chosen parameters as a one-argument function for the solver.
quartic_function = lambda q: quartic_potential(q, 1.0, quartic_coupling)

# Solve the interacting Hamiltonian with the same domain and resolution conventions.
(
    quartic_grid_df,
    quartic_hamiltonian,
    quartic_h,
    quartic_energies,
    quartic_vectors,
    quartic_wavefunctions,
) = solve_low_energy_states(7.0, 500, quartic_function, state_count=6)

# Compare the interacting spectrum with the exactly soluble harmonic limit.
quartic_energy_df = pd.DataFrame({
    "state_n": np.arange(6),
    "harmonic_energy": np.arange(6) + 0.5,
    "quartic_energy": quartic_energies,
})
# Subtract the harmonic level to isolate the interaction-induced shift.
quartic_energy_df["interaction_shift"] = (
    quartic_energy_df["quartic_energy"] - quartic_energy_df["harmonic_energy"]
)
quartic_energy_df

In [ ]:
# Plot the potential with probability densities offset vertically by their energies.
quartic_plot_df = quartic_grid_df[["q", "potential"]].copy()

# Draw the potential first; the state densities will be overlaid on this axis.
fig, ax = plt.subplots()
ax.plot(
    quartic_plot_df["q"],
    quartic_plot_df["potential"],
    color="black",
    label=r"$V(q)$",
)

# Display only the first four states to keep the figure readable.
for n in range(4):
    # Scaling the density is only for display; the vertical offset is the energy.
    quartic_plot_df[f"offset_density_{n}"] = (
        quartic_energies[n] + 0.55 * quartic_wavefunctions[:, n] ** 2
    )
    # Plot each shifted density using its DataFrame column.
    ax.plot(
        quartic_plot_df["q"],
        quartic_plot_df[f"offset_density_{n}"],
        label=fr"$E_{n}+0.55|\psi_{n}|^2$",
    )
    ax.axhline(quartic_energies[n], color="gray", linewidth=0.6, alpha=0.6)

# Restrict the view to the classically relevant portion of the potential.
ax.set(
    xlim=(-4, 4),
    ylim=(0, 8),
    xlabel=r"$q$",
    ylabel="dimensionless energy",
    title=fr"Quartic oscillator with $\lambda={quartic_coupling}$",
)
ax.legend(ncols=2, fontsize=8)
plt.show()

## 5. Two convergence controls: box size and grid spacing

The numerical model has two independent approximations:

1. **Domain truncation:** finite $q_{\max}$ replaces decay at infinity with $\psi(\pm q_{\max})=0$.
2. **Derivative discretization:** finite $h=2q_{\max}/N$ approximates $d^2/dq^2$ with a centered difference.

Changing $q_{\max}$ while also changing $h$ confounds these effects. A useful convergence study varies one while holding the other approximately fixed.

For domain convergence, monitor probability near the edge:

$$
P_{\mathrm{edge}}\approx h\sum_{|q_i|>0.9q_{\max}}|\psi_0(q_i)|^2.
$$

For grid convergence, compare energies at fixed $q_{\max}$. The centered second derivative should give leading energy errors proportional to $h^2$ once boundary error is negligible.

## In-class Activity 2 - Separate domain and grid errors (14 minutes)

Use separate numerical references so that each study isolates one approximation. Then perform two studies:

- **domain study:** vary $q_{\max}=2,2.5,3,3.5,4$ while keeping $h=0.025$, and compare with a wider box at the same $h$;
- **grid study:** vary $N=150,300,500,700$ while keeping $q_{\max}=6$.

For each solve, record $E_0$, $|E_0-E_{0,\mathrm{ref}}|$, and $P_{\mathrm{edge}}$. Plot the domain-study error against $q_{\max}$ and the grid-study error against $h$.

**Discuss before coding:** Why can decreasing $h$ fail to improve the result when $q_{\max}$ is too small? How would the memory cost change if this dense matrix dimension doubled?

In [ ]:
# Match h in the wide-box reference so grid error cancels in the domain study.
domain_reference_result = solve_low_energy_states(8.0, 640, quartic_function, state_count=1)
domain_reference_energy = domain_reference_result[3][0]

# Use a wider, finer reference for the grid study; it is numerical, not exact.
grid_reference_result = solve_low_energy_states(8.0, 1000, quartic_function, state_count=1)
grid_reference_energy = grid_reference_result[3][0]

# Accumulate both studies in one tidy table, distinguished by a label.
convergence_rows = []

# Keep h fixed to isolate finite-domain effects.
for q_max_i in [2.0, 2.5, 3.0, 3.5, 4.0]:
    # Choose N so that h remains exactly 0.025 as q_max changes.
    interval_count_i = int(round(2.0 * q_max_i / 0.025))

    # TODO: solve for the ground state at this q_max and interval count.
    result_i = ...

    # Unpack only the quantities needed for convergence diagnostics.
    grid_i, h_i, energy_i, wavefunction_i = (
        result_i[0], result_i[2], result_i[3][0], result_i[5][:, 0]
    )
    # Mark the outer 10% of this particular computational domain.
    edge_i = np.abs(grid_i["q"]) > 0.9 * q_max_i

    # TODO: compute the probability in the outer 10% of the box.
    edge_probability_i = ...

    # Compare with the fixed-spacing wide-box reference.
    convergence_rows.append({
        "study": "domain",
        "q_max": q_max_i,
        "interval_count": interval_count_i,
        "h": h_i,
        "energy": energy_i,
        "absolute_error": abs(energy_i - domain_reference_energy),
        "edge_probability": edge_probability_i,
    })

# Keep q_max fixed to isolate finite-grid-spacing effects.
for interval_count_i in [150, 300, 500, 700]:
    # Hold the physical box fixed so only h changes.
    q_max_i = 6.0

    # TODO: solve for the ground state at this fixed domain and grid size.
    result_i = ...

    # Unpack only the quantities needed for convergence diagnostics.
    grid_i, h_i, energy_i, wavefunction_i = (
        result_i[0], result_i[2], result_i[3][0], result_i[5][:, 0]
    )
    # Mark the outer 10% of this particular computational domain.
    edge_i = np.abs(grid_i["q"]) > 0.9 * q_max_i
    edge_probability_i = h_i * np.sum(np.abs(wavefunction_i[edge_i]) ** 2)

    # Compare with the fine-grid reference on a sufficiently wide domain.
    convergence_rows.append({
        "study": "grid",
        "q_max": q_max_i,
        "interval_count": interval_count_i,
        "h": h_i,
        "energy": energy_i,
        "absolute_error": abs(energy_i - grid_reference_energy),
        "edge_probability": edge_probability_i,
    })

# One DataFrame makes selecting, plotting, and reporting either study straightforward.
convergence_df = pd.DataFrame(convergence_rows)
convergence_df

In [ ]:
# TODO: select the domain-study and grid-study rows from convergence_df.
domain_df = ...
# TODO: select the grid-study rows from convergence_df.
grid_df = ...

# Place domain and grid convergence side by side for comparison.
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

# TODO: plot domain-study absolute_error versus q_max on a logarithmic y-axis.
...

# TODO: plot grid-study absolute_error versus h on log-log axes.
...

# Add physical axis labels only after the student-generated curves exist.
axes[0].set(xlabel=r"$q_{\max}$", ylabel=r"$|E_0-E_{0,\mathrm{ref}}|$",
            title="Finite-domain convergence")
axes[1].set(xlabel=r"grid spacing $h$", ylabel=r"$|E_0-E_{0,\mathrm{ref}}|$",
            title="Finite-difference convergence")
# Render both panels together.
plt.show()

## 6. Worked example: a nearly degenerate double well

First consider the symmetric potential

$$
V(q)=\frac{1}{4}(q^2-v^2)^2,
\qquad v=2.2.
$$

Classically, the minima at $q=\pm v$ are exactly degenerate. In the finite quantum problem, the wavefunction can tunnel through the central barrier. The two lowest states are therefore an even ground state and an odd first excited state separated by the small but nonzero tunneling gap

$$
\Delta E=E_1-E_0.
$$

Because the potential has parity symmetry, both energy eigenstates should satisfy $\langle q\rangle=0$. We first solve and inspect this symmetric problem before introducing any perturbation.

In [ ]:
# v fixes the two classical minima at q = +/-v.
double_well_v = 2.2


def symmetric_double_well(q):
    """Return the symmetric quartic double-well potential."""
    # Squaring q^2-v^2 gives two minima and a central tunneling barrier.
    return 0.25 * (q**2 - double_well_v**2) ** 2


# Evaluate the potential on a smooth grid used only for visualization.
potential_q = np.linspace(-4.0, 4.0, 600)
double_well_potential_df = pd.DataFrame({"q": potential_q})
double_well_potential_df["potential"] = symmetric_double_well(potential_q)

# The minima have V=0, while the barrier height is V(0)=v^4/4.
barrier_height = symmetric_double_well(np.array([0.0]))[0]

fig, ax = plt.subplots()
ax.plot(
    double_well_potential_df["q"],
    double_well_potential_df["potential"],
    color="black",
    label=r"$V(q)=\frac{1}{4}(q^2-v^2)^2$",
)

# Mark the classical stationary points to orient the quantum calculation.
ax.scatter(
    [-double_well_v, double_well_v],
    [0.0, 0.0],
    color="tab:blue",
    zorder=3,
    label=r"degenerate minima $q=\pm v$",
)
ax.scatter(
    [0.0],
    [barrier_height],
    color="tab:red",
    zorder=3,
    label=fr"central barrier $V(0)={barrier_height:.2f}$",
)

ax.set(
    xlim=(-4, 4),
    ylim=(-0.25, 8),
    xlabel=r"$q$",
    ylabel=r"$V(q)$",
    title="Symmetric double-well potential",
)
ax.legend()
plt.show()

In [ ]:
# Resolve the tunneling doublet and two additional excited states.
symmetric_result = solve_low_energy_states(
    7.0, 800, symmetric_double_well, state_count=4
)

# Unpack the grid, spacing, energies, and continuum-normalized wavefunctions.
symmetric_grid_df = symmetric_result[0]
double_well_h = symmetric_result[2]
symmetric_energies = symmetric_result[3]
symmetric_wavefunctions = symmetric_result[5]
q_values = symmetric_grid_df["q"].to_numpy()

# The difference between the first two energies is the tunneling gap.
tunneling_splitting = symmetric_energies[1] - symmetric_energies[0]

# Check localization and parity for each member of the doublet.
symmetric_rows = []
for n in (0, 1):
    wavefunction = symmetric_wavefunctions[:, n]
    q_mean = double_well_h * np.sum(q_values * np.abs(wavefunction) ** 2)
    parity = double_well_h * np.sum(
        np.conj(wavefunction) * wavefunction[::-1]
    ).real

    symmetric_rows.append({
        "state_n": n,
        "energy": symmetric_energies[n],
        "q_mean": q_mean,
        "parity_expectation": parity,
    })

symmetric_summary_df = pd.DataFrame(symmetric_rows)

# Store the two probability densities in a DataFrame before plotting.
symmetric_density_df = symmetric_grid_df[["q"]].copy()
for n in (0, 1):
    symmetric_density_df[f"density_{n}"] = (
        np.abs(symmetric_wavefunctions[:, n]) ** 2
    )

# Both parity eigenstates have probability in both classical wells.
fig, ax = plt.subplots()
for n in (0, 1):
    ax.plot(
        symmetric_density_df["q"],
        symmetric_density_df[f"density_{n}"],
        label=fr"$|\psi_{n}|^2$",
    )

# Mark the classical minima from the potential figure as spatial references.
ax.axvline(-double_well_v, color="gray", linestyle="--", linewidth=0.8)
ax.axvline(double_well_v, color="gray", linestyle="--", linewidth=0.8)
ax.set(
    xlim=(-4, 4),
    xlabel=r"$q$",
    ylabel=r"probability density $|\psi|^2$",
    title="Lowest states of the symmetric double well",
)
ax.legend()
plt.show()

print(f"Tunneling gap Delta E = {tunneling_splitting:.3e}")
symmetric_summary_df

### Extension: add a small symmetry-breaking tilt

Now perturb the solved double well by

$$
\delta V(q)=\epsilon q,
\qquad \epsilon=10^{-3}.
$$

The potential change is small compared with the barrier, but the approximate energy bias between the classical minima is

$$
\Delta V_{\mathrm{wells}}\approx2\epsilon v.
$$

For these parameters, $2\epsilon v$ is larger than the tunneling gap found above. We should therefore expect the new energy eigenstates to become localized mostly in opposite wells rather than remain parity eigenstates.

In [ ]:
# Choose a tilt small relative to the barrier but compare it with the doublet gap.
epsilon = 1.0e-3


def tilted_double_well(q):
    """Add a weak linear field to the previously solved symmetric potential."""
    return symmetric_double_well(q) + epsilon * q


# Use the same box and grid so changes can be attributed to the perturbation.
tilted_result = solve_low_energy_states(
    7.0, 800, tilted_double_well, state_count=4
)

# Extract the perturbed spectrum and continuum-normalized wavefunctions.
tilted_energies = tilted_result[3]
tilted_wavefunctions = tilted_result[5]
classical_well_bias = 2.0 * epsilon * double_well_v

# Compare energies and position expectation values before and after the tilt.
comparison_rows = []
for n in (0, 1):
    symmetric_q_mean = double_well_h * np.sum(
        q_values * np.abs(symmetric_wavefunctions[:, n]) ** 2
    )
    tilted_q_mean = double_well_h * np.sum(
        q_values * np.abs(tilted_wavefunctions[:, n]) ** 2
    )

    comparison_rows.append({
        "state_n": n,
        "symmetric_energy": symmetric_energies[n],
        "tilted_energy": tilted_energies[n],
        "symmetric_q_mean": symmetric_q_mean,
        "tilted_q_mean": tilted_q_mean,
    })

tilt_comparison_df = pd.DataFrame(comparison_rows)

# Store the perturbed densities in a DataFrame before plotting them.
tilted_plot_df = symmetric_grid_df[["q"]].copy()
for n in (0, 1):
    tilted_plot_df[f"density_{n}"] = np.abs(tilted_wavefunctions[:, n]) ** 2

fig, ax = plt.subplots()
for n in (0, 1):
    ax.plot(
        tilted_plot_df["q"],
        tilted_plot_df[f"density_{n}"],
        label=fr"tilted $|\psi_{n}|^2$",
    )

# Mark the unperturbed classical minima as spatial reference points.
ax.axvline(-double_well_v, color="gray", linestyle="--", linewidth=0.8)
ax.axvline(double_well_v, color="gray", linestyle="--", linewidth=0.8)
ax.set(
    xlim=(-4, 4),
    xlabel=r"$q$",
    ylabel=r"probability density $|\psi|^2$",
    title=fr"Localized states after a small tilt: $\epsilon={epsilon}$",
)
ax.legend()
plt.show()

print(f"Tunneling gap Delta E = {tunneling_splitting:.3e}")
print(f"Classical well bias 2 epsilon v = {classical_well_bias:.3e}")
tilt_comparison_df

### Interpretation: the spectral gap controls eigenvector sensitivity

The symmetric eigenstates have $\langle q\rangle\approx0$ and parity $+1$ or $-1$. After the tilt, the two lowest states acquire large negative or positive $\langle q\rangle$, confirming that they are localized mostly in opposite wells.

For a Hermitian Hamiltonian, eigenvalues are stable under a perturbation $\delta H$ in the sense that

$$
|\delta E_n|\leq\|\delta H\|_2.
$$

Individual eigenvectors can nevertheless rotate strongly when the neighboring spectral gap is small:

$$
\sin\theta_n\lesssim
\frac{\|\delta H\|_2}{\operatorname{gap}_n}.
$$

The low-energy **subspace** remains meaningful even when the basis vectors within a nearly degenerate doublet change substantially.

This double well is a useful finite-dimensional analogue of potentials used when discussing spontaneous symmetry breaking. The analogy has a limit: a finite one-dimensional quantum system retains a parity-even exact ground state and a nonzero tunneling gap. Exactly degenerate vacua require an appropriate infinite-volume or infinite-degree-of-freedom limit.

## Workflow checkpoint

For a numerical quantum eigenproblem, document:

- the units or nondimensionalization of the Hamiltonian;
- the finite domain, grid spacing, and boundary conditions;
- a numerical check that $H=H^\dagger$ and the eigensolver used;
- whether the representation is dense, sparse, banded, or block structured;
- continuum normalization and phase/sign conventions;
- eigenpair residuals and orthogonality checks;
- symmetry quantum numbers such as parity;
- separate domain-size and grid-spacing convergence studies;
- the spectral gaps relevant to interpreting individual eigenvectors.

A table of energy eigenvalues without these checks is not yet a numerical physics result.

## Takeaways

- A quantum Hamiltonian is Hermitian; its detailed matrix pattern depends on the chosen basis and discretization.
- `eigh` solves the full Hermitian matrix without relying on that special pattern.
- The harmonic oscillator provides an exact end-to-end benchmark before adding an interaction.
- Discrete eigenvectors must be rescaled to represent continuum-normalized wavefunctions.
- Small residuals verify the matrix solve; domain and grid convergence verify the continuum approximation.
- Near-degenerate eigenvectors are sensitive to perturbations on the scale of their spectral gap, even when the low-energy subspace remains stable.
- In a finite symmetric double well, the exact ground state has definite parity; localized states are combinations of a nearly degenerate doublet.

## Open-source references and further reading

- [Unit 3 numerical-methods resources](../resources/unit-3-numerical-methods.md)
- [SciPy `eigh` documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.eigh.html)
- [LAPACK Users' Guide: symmetric and Hermitian eigenproblems](https://www.netlib.org/lapack/lug/node48.html) - matrix reduction to tridiagonal form, solution of the reduced problem, and eigenvector back-transformation
- [SciPy sparse Hermitian eigensolver `eigsh`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.eigsh.html)
- [NumPy linear algebra documentation](https://numpy.org/doc/stable/reference/routines.linalg.html)
- [MIT OpenCourseWare 8.06, Quantum Physics III](https://ocw.mit.edu/courses/8-06-quantum-physics-iii-spring-2018/)
- [David Tong, *Quantum Field Theory* lecture notes](https://www.damtp.cam.ac.uk/user/tong/qft.html) - especially the free scalar field as oscillator modes and scalar-field interactions
- [*Fundamentals of Numerical Computation*, eigenvalue problems](https://fncbook.com/chapter7-1/)

Natural extensions include matrix representations in a harmonic-oscillator basis, multidimensional sparse Hamiltonians, imaginary-time evolution, and Lanczos methods for many-body systems.